# Parallel Endpoint Deployment for Baseline Evaluation

This notebook contains the updated code for deploying SageMaker endpoints in parallel rather than sequentially. This approach is more efficient and reduces the overall time needed for model deployment.

## How to use this notebook:
1. Run notebook 02_baseline_evaluation.ipynb up to section 5 (Helper Functions)
2. Switch to this notebook and run the cells to deploy endpoints in parallel
3. Return to notebook 02_baseline_evaluation.ipynb and continue from section 7 (Measure Baseline Metrics)

## Import Dependencies

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
import re
from sagemaker.huggingface import HuggingFaceModel
from sagemaker import get_execution_role
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM
from IPython.display import clear_output

## Load Workshop Settings and Model Information

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE
%store -r ENDPOINT_INSTANCE_TYPE
%store -r model_info

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Endpoint Instance Type: {ENDPOINT_INSTANCE_TYPE}")
    print(f"Loaded information for {len(model_info)} models")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    print("Then run notebook 02_baseline_evaluation.ipynb up to section 5.")

## Helper Function for Name Sanitization

In [ ]:
def sanitize_name(name):
    """Sanitize a name to be used as part of an endpoint name.
    Endpoint names must satisfy regex pattern: [a-zA-Z0-9](-*[a-zA-Z0-9]){0,62}
    """
    # Replace underscores with hyphens
    sanitized = name.replace('_', '-')
    # Replace any other invalid characters with hyphens
    sanitized = re.sub(r'[^a-zA-Z0-9-]', '-', sanitized)
    # Ensure it doesn't start or end with a hyphen
    sanitized = sanitized.strip('-')
    # Ensure no consecutive hyphens
    sanitized = re.sub(r'-+', '-', sanitized)
    return sanitized

## Deploy Models to SageMaker Endpoints in Parallel

This updated approach deploys all models in parallel, which is more efficient than deploying them sequentially.

In [ ]:
# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Dictionary to store endpoint names and deployment status
endpoint_names = {}
endpoint_status = {}

# First, prepare all model configurations
print("Preparing model configurations...")
model_configs = {}

for model_key, info in model_info.items():
    print(f"Preparing configuration for {model_key} model...")
    
    # Create a unique endpoint name with sanitized model key
    endpoint_name = f"model-opt-workshop-{sanitize_name(model_key)}-{int(time.time())}"
    endpoint_names[model_key] = endpoint_name
    endpoint_status[model_key] = "Preparing"
    
    # Create environment variables for the Hugging Face model
    env = {
        'HF_MODEL_ID': info["hub_model_id"],
        'HF_TASK': info["task"]
    }
    
    # Create a Hugging Face model
    huggingface_model = HuggingFaceModel(
        model_data=None,  # No model data, will use HF_MODEL_ID instead
        role=SAGEMAKER_ROLE_ARN,
        transformers_version="4.26",
        pytorch_version="1.13",
        py_version="py39",
        env=env
    )
    
    # Store the model configuration
    model_configs[model_key] = huggingface_model
    print(f"Configuration prepared for {model_key} model")

In [ ]:
# Now deploy all models in parallel
print("\nDeploying all models in parallel...")
deployment_futures = {}

for model_key, model in model_configs.items():
    print(f"Starting deployment for {model_key} model...")
    endpoint_status[model_key] = "Deploying"
    
    # Deploy the model to an endpoint asynchronously
    deployment_futures[model_key] = model.deploy(
        initial_instance_count=1,
        instance_type=ENDPOINT_INSTANCE_TYPE,
        endpoint_name=endpoint_names[model_key],
        wait=False  # Don't wait for deployment to complete
    )
    
    print(f"Deployment started for {model_key} model to endpoint: {endpoint_names[model_key]}")

In [ ]:
# Monitor deployment status
print("\nMonitoring deployment status...")

def check_endpoint_status():
    """Check the status of all endpoints being deployed."""
    sagemaker_client = boto3.client('sagemaker')
    statuses = {}
    
    for model_key, endpoint_name in endpoint_names.items():
        try:
            response = sagemaker_client.describe_endpoint(
                EndpointName=endpoint_name
            )
            statuses[model_key] = response['EndpointStatus']
        except Exception as e:
            statuses[model_key] = f"Error: {str(e)}"
    
    return statuses

# Wait for all deployments to complete
all_completed = False
while not all_completed:
    current_statuses = check_endpoint_status()
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current deployment statuses:")
    for model_key, status in current_statuses.items():
        print(f"{model_key}: {status}")
    
    # Check if all deployments are completed
    all_completed = all(status in ["InService", "Failed"] for status in current_statuses.values())
    
    if not all_completed:
        print("\nWaiting for 30 seconds before checking again...")
        time.sleep(30)

print("\nAll deployments have completed.")

In [ ]:
# Store endpoint names for later use
%store endpoint_names

print("Endpoint names have been stored. You can now return to notebook 02_baseline_evaluation.ipynb")
print("and continue from section 7 (Measure Baseline Metrics).")